# Lab — FlashAttention on a real LLM

**GPU Mastery · CUDA** · runs on the **2× RTX 5090** server *and* a **Colab GPU** (auto-scales to your VRAM)

Standard attention builds an **N×N** scores matrix (memory grows as **N²**); **FlashAttention** streams it through on-chip memory (memory grows as **N**). You'll prove it with the GPU's own memory counter — first on raw attention (works on any GPU), then inside a real LLM.

## 0. Requirements & one important caveat

`torch` (Blackwell = CUDA 12.8), `transformers`, `accelerate`. We use PyTorch's built-in **`scaled_dot_product_attention`** — no `flash-attn` install needed.

> ⚠️ **The real FlashAttention-2 kernel needs an Ampere GPU (sm_80) or newer** — A100, L4, RTX 30/40/50-series, your **5090**. On an older **Colab T4 (Turing)**, SDPA falls back to a slower backend, so **Part 2's LLM gap will look small on a T4** — but **Part 1 shows the effect on any GPU**, and the full effect appears on your 5090.
>
> **On Colab:** Runtime → Change runtime type → **GPU**. The notebook auto-picks smaller sizes for a 16 GB card.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # reduce fragmentation OOMs
import torch, time, math, gc
import torch.nn.functional as F

assert torch.cuda.is_available(), "No GPU — enable a GPU runtime"
p = torch.cuda.get_device_properties(0)
TOTAL_GB = p.total_memory / 1e9
SMALL = TOTAL_GB < 24         # T4/L4 vs 5090/A100
print(f"GPU: {p.name}  {TOTAL_GB:.1f} GB  (sm_{p.major}{p.minor})  | torch {torch.__version__}")
print("mode:", "SMALL card — using shorter sequences" if SMALL else "BIG card — using long sequences")
if p.major < 8:
    print("NOTE: this GPU is pre-Ampere (sm_80) — SDPA can't use the FlashAttention-2 kernel; Part 1 still proves the point.")

def clear():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_gb():
    return torch.cuda.max_memory_allocated() / 1e9

## 1. The memory wall, in isolation  *(works on any GPU)*

Fake Q, K, V for **32 heads × head-dim 128**. Run attention two ways at growing length:
- **`naive`** — actually forms `S = Q·Kᵀ` (the N×N matrix), softmax, `·V`
- **`flash`** — `F.scaled_dot_product_attention` (never materialises S)

Watch **naive** track the theoretical N×N size, quadruple each doubling, and **OOM** — while **flash** stays flat.

In [ ]:
def bench_attn(seq, heads=32, dim=128, mode="flash"):
    clear()
    try:
        q = torch.randn(1, heads, seq, dim, device="cuda", dtype=torch.float16)
        k = torch.randn_like(q); v = torch.randn_like(q)
        torch.cuda.synchronize(); t = time.time()
        if mode == "naive":
            s = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(dim)); pr = s.softmax(-1); o = pr @ v
            del s, pr, o
        else:
            o = F.scaled_dot_product_attention(q, k, v); del o
        torch.cuda.synchronize(); dt = (time.time() - t) * 1e3; m = peak_gb()
        del q, k, v; clear(); return m, dt
    except torch.cuda.OutOfMemoryError:
        clear(); return None, None

seqs = [1024, 2048, 4096, 8192, 16384] if SMALL else [2048, 4096, 8192, 16384, 32768]
print(f"{'seq':>7} | {'N×N (theory)':>13} | {'naive peak':>11} {'ms':>7} | {'flash peak':>11} {'ms':>7}")
for n in seqs:
    theory = 32 * n * n * 2 / 1e9
    nm, nt = bench_attn(n, mode="naive")
    fm, ft = bench_attn(n, mode="flash")
    ns = f"{nm:8.2f} GB {nt:6.0f}" if nm else f"{'OOM':>11} {'—':>6}"
    fs = f"{fm:8.2f} GB {ft:6.0f}" if fm else f"{'OOM':>11} {'—':>6}"
    print(f"{n:>7} | {theory:10.2f} GB | {ns} | {fs}")

**What you should see (any GPU):** the **naive** peak ≈ the theoretical N×N and **quadruples each time you double the sequence**, then **OOMs**. The **flash** peak barely moves. Same maths, same answer — the only difference is whether the big matrix touches memory. *This is the whole lesson; Parts 2–3 show it inside a model.*

## 2. Inside a real LLM

Same effect in an actual model, via `attn_implementation`: **`"eager"`** (plain) vs **`"sdpa"`** (FlashAttention backend on Ampere+). We measure **prefill** peak memory + time, stopping each run at its first OOM.

In [ ]:
# ===== CONFIG =====
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# ==================
from transformers import AutoModelForCausalLM, AutoTokenizer

def run_impl(impl, seqs):
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16,
                                                 attn_implementation=impl, device_map={"": 0}).eval()
    rows = []
    for seq in seqs:
        clear()
        try:
            ids = torch.randint(0, tok.vocab_size, (1, seq), device="cuda")
            torch.cuda.synchronize(); t = time.time()
            with torch.no_grad(): model(ids)
            torch.cuda.synchronize()
            rows.append((seq, peak_gb(), (time.time() - t) * 1e3)); del ids
        except torch.cuda.OutOfMemoryError:
            rows.append((seq, None, None)); clear(); break     # stop this impl at its first OOM
    del model; clear(); return rows

seqs = [512, 1024, 2048, 4096, 8192] if SMALL else [1024, 2048, 4096, 8192, 16384]
eager = dict((s, (m, t)) for s, m, t in run_impl("eager", seqs))
sdpa  = dict((s, (m, t)) for s, m, t in run_impl("sdpa",  seqs))

print(f"{'context':>8} | {'eager peak':>11} {'ms':>7} | {'flash(sdpa) peak':>16} {'ms':>7}")
for s in seqs:
    em, et = eager.get(s, (None, None)); fm, ft = sdpa.get(s, (None, None))
    es = f"{em:8.2f} GB {et:6.0f}" if em else f"{'OOM':>11} {'—':>6}"
    fs = f"{fm:8.2f} GB {ft:6.0f}" if fm else f"{'OOM':>16} {'—':>6}"
    print(f"{s:>8} | {es} | {fs}")

**Reading it:** on an **Ampere+ GPU (your 5090)** the `sdpa`/flash column uses clearly less memory and OOMs at a *longer* context than `eager` — that extra headroom is what serves long prompts. On a **T4** the two look similar (SDPA can't use the Flash kernel there) — so trust **Part 1** for the clean proof and this part for the *frontier* (where does eager OOM first).

## 3. Push the context — find the longest that fits

Generate from an increasingly long prompt with the flash backend; step down until it fits. Bigger cards reach much further.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16,
                                             attn_implementation="sdpa", device_map={"": 0}).eval()

targets = [4096, 3072, 2048, 1024] if SMALL else [16384, 12288, 8192, 4096]
for ctx in targets:
    clear()
    try:
        ids = torch.randint(0, tok.vocab_size, (1, ctx), device="cuda")
        torch.cuda.synchronize(); t = time.time()
        with torch.no_grad(): out = model.generate(ids, max_new_tokens=16, do_sample=False)
        torch.cuda.synchronize()
        print(f"✅ generated from a {ctx}-token prompt  |  peak {peak_gb():.2f} GB  |  {time.time()-t:.2f} s")
        del ids, out; break
    except torch.cuda.OutOfMemoryError:
        print(f"❌ {ctx} tokens: OOM — trying smaller"); clear()
del model; clear()

## Reflection (write your answers)

1. In Part 1, how closely did **naive peak** match the **theoretical N×N**? Where did naive OOM while flash didn't?
2. Each time you **doubled** the sequence, what happened to naive memory — does it match "grows as N²"?
3. What GPU are you on (`sm_XX`)? If it's a T4, why did Part 2 look flat — and what would change on the 5090?
4. Your 5090 has 32 GB. From Part 1's theory column, what's the longest context a **naive** 32-head attention could hold — and does FlashAttention remove that ceiling?

### Cleanup

In [ ]:
gc.collect(); torch.cuda.empty_cache()
print("done — GPU memory released")